# Quickstart (synthetic mobility data)

End-to-end pipeline run using synthetic mobility data (assumes you have raw ACS and ATUS data as in the examples).

# 0. SETUP (START HERE)

## 0.1. Imports

In [1]:
from pathlib import Path

import pandas as pd
from collections import defaultdict

from mobcalibrate import Calibrator, CalibrationResult
from helpers import acs, atus, mobility
from helpers import prep_calibration_inputs as prep


## 0.2. CONFIGS: CBSA Code & Paths

In [2]:
CBSA_CODE = 38060

DATA_DIR = Path('data')
OUTPUT_DIR = Path('data/processed')

ACS_INCOME_FILE = DATA_DIR / f'acs/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B19001-Data.csv'
ACS_AGE_FILE    = DATA_DIR / f'acs/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B01001-Data.csv'

RESP_FILE = DATA_DIR / 'atus/atusresp-0324/atusresp_0324.dat'
CPS_FILE  = DATA_DIR / 'atus/atuscps-0324/atuscps_0324.dat'
ACT_FILE  = DATA_DIR / 'atus/atusact-0324/atusact_0324.dat'

# Synthetic mobility user-day sequences (CSV): first column is a row index (used as user_id),
# then 48 sequence columns named 0..47, then GEOID.
MOBILITY_SEQ_FILE = DATA_DIR / 'mobility/synthetic_mobility.csv'

## 0.3. CONFIGS: ACS Processing

In [3]:
# Age group definitions 
# (if None, use acs.DEFAULT_AGE_GROUPS)
AGE_GROUPS = {
    "<18": [
        "Under 5 years", 
        "5 to 9 years",
        "10 to 14 years", 
        "15 to 17 years",
    ],

    "18-24": [
        "18 and 19 years", 
        "20 years",
        "21 years", 
        "22 to 24 years",
    ],

    "25-44": [
        "25 to 29 years", 
        "30 to 34 years",
        "35 to 39 years", 
        "40 to 44 years",
    ],

    "45-66": [
        "45 to 49 years", 
        "50 to 54 years",
        "55 to 59 years", 
        "60 and 61 years",
        "62 to 64 years", 
        "65 and 66 years",
    ],

    "67+": [
        "67 to 69 years", 
        "70 to 74 years",
        "75 to 79 years", 
        "80 to 84 years",
        "85 years and over",
    ],
}
DROP_AGE_GROUPS = ['<18']      # drop groups not represented in your mobility sample


# Income group mapping
# Leave None to auto-derive quartiles  from the CBSA-level row of the B19001 table.
# Otherwise, pass an explicit dict to override (see walkthrough_02_process_acs.ipynb for example).
INCOME_GROUP_MAPPING = None


## 0.4. CONFIGS: ATUS Processing

In [4]:
# years to pool data
YEARS = list(range(2004, 2020))
# minimum age of respondents
AGE_MIN = 18

# Step 1: raw TEWHERE code -> intermediate location code (1-8).
# Unlisted TEWHERE codes (modes of transportation/travel) -> 3 (unspecified place).
TEWHERE_MAP = defaultdict(lambda: 3, {
    1:  1,   # home
    2:  2,   # work
    3:  3,   # someone else's home -> unspecified place
    4:  4,   # restaurants/bars
    5:  3,   # place of worship    -> unspecified place
    6:  5,   # grocery store       -> stores/mall
    7:  5,   # other store/mall    -> stores/mall
    8:  6,   # school
    9:  7,   # outdoors away from home
    31: 8,   # gym/health club     -> other poi
    10: 8,   # library             -> other poi
    11: 8,   # other place         -> other poi
    30: 8,   # bank                -> other poi
    32: 8,   # post office         -> other poi
    89: 3,   # unspecified         -> unspecified place
})
# intermediate code labels (for reference):
# 1: home, 2: work, 3: unspecified place, 4: restaurants/bars,
# 5: stores/mall, 6: school, 7: outdoors, 8: other poi

# Step 2: intermediate code -> final alphabet (must match SEQUENCE_METRIC_SPECS['all_labels']).
# Applied after modal downsampling to T-minute cells, so the modal state is chosen
# among the 8 intermediate codes (ties -> lowest code, i.e. home > work > unspecified > POIs).
COLLAPSE_MAP = defaultdict(lambda: 'Unspecified place', {
    1: 'Home',
    2: 'Work',
    3: 'Unspecified place',
    4: 'POI',   # restaurants/bars
    5: 'POI',   # stores/mall
    6: 'POI',   # school
    7: 'POI',   # outdoors
    8: 'POI',   # other poi
})

# sequence cell width in minutes
T = 30

# number of behavioral clusters
K = 4

# respondent weight column name  
WEIGHT_COL = 'TUFINLWGT'

# stratification column names
ROW_VAR = 'income'
COL_VAR = 'age'

## 0.5. CONFIGS: Sequence Metrics & Distances

In [5]:
# specify alphabet labels present in the sequences
# as well as the subset of the metrics to use in distance calculations
SEQUENCE_METRIC_SPECS = {
    'all_labels': ['Home', 'Work', 'Unspecified place', 'POI'],
    'feature_subset': 'all',   # 'all' | 'metrics_only' | 'tu_only' | 'edges_only'
    'normalize': True,        # rescale features with normalize_sequence_metrics() before distances
}

# Column names in your mobility sequence DataFrame that contain the activity cells
# There should be 24*60/T of them (e.g. 48 cells for T=30 minutes).
MOB_SEQUENCE_COLS = [str(n) for n in range(int(24 * 60 / T))]

## 0.5. CONFIGS: Mobility Sequence Cluster Assignment

In [6]:
# number of nearest ATUS neighbors to find in the KNN cluster assignment
KNN_K = 10
# threshold for the number of neighbors in agreement 
# in order to assign a mobility user to cluster
KNN_THRESHOLD = 0.5

## 0.7. CONFIGS: Calibration

In [7]:
# number of replicates to run for variance estimation
# (each replicate resamples demographic attributes)
NUM_REPLICATES = 51

# seed number for calibration (used in sampling)
SEED = 1234

# calibration mode
# one of: 'uniform' | 'demographic' | 'behavioral' | 'demographic_behavioral'
CALIBRATION_MODE = 'demographic_behavioral'


# 1. Process ACS

Produces a CBG-level joint (income × age) distribution and CBSA-level margins. Income groups auto-derive as quartiles from the CBSA row (printed below); age groups come from `AGE_GROUPS` or the default.


In [8]:
acs_cbg_distr, income_margin, age_margin = acs.process_cbsa(
    ACS_INCOME_FILE, ACS_AGE_FILE,
    cbsa_code=CBSA_CODE,
    income_group_mapping=INCOME_GROUP_MAPPING,
    age_groups=AGE_GROUPS,
    drop_age_groups=DROP_AGE_GROUPS,
    row_var=ROW_VAR,
    col_var=COL_VAR,
)

print('acs_cbg_distr shape:', acs_cbg_distr.shape)
print('income margin:')
print(income_margin)
print('age margin:')
print(age_margin)
acs_cbg_distr.head()


acs.process_cbsa: auto-derived income quartile mapping: ['<35k', '35k-75k', '75k-125k', '125k+']
acs_cbg_distr shape: (2987, 9)
income margin:
     income       pop
0      <35k  416253.0
1   35k-75k  547471.0
2  75k-125k  410402.0
3     125k+  371093.0
age margin:
     age        pop
0  18-24   444363.0
1  25-44  1337601.0
2  45-66  1262610.0
3    67+   663574.0


,GEOID,18-24,25-44,45-66,67+,<35k,35k-75k,75k-125k,125k+
0,040130101021,0.003542,0.319953,0.279811,0.396694,0.085106,0.12766,0.378251,0.408983
1,040130101022,0.006592,0.071852,0.555043,0.366513,0.295756,0.087533,0.098143,0.518568
2,040130101023,0.023019,0.0,0.464151,0.51283,0.149068,0.10352,0.089027,0.658385
3,040130101031,0.045374,0.129004,0.55694,0.268683,0.22651,0.275168,0.20302,0.295302
4,040130101032,0.026382,0.236809,0.499372,0.237437,0.118758,0.140351,0.273954,0.466937


# 2. Process ATUS

Loads diaries, stratifies respondents against the ACS margins from Section 1, generates sequences and clusters them into K behavioral clusters via weighted k-medoids using `SEQUENCE_METRIC_SPECS`.


In [9]:
print('Loading ATUS diaries...')
resp, diaries = atus.load(
    RESP_FILE, CPS_FILE, ACT_FILE, TEWHERE_MAP,
    years=YEARS, age_min=AGE_MIN, cbsa_filter=[CBSA_CODE],
)

print('Stratifying respondents based on ACS targets...')
atus_meta, group_maps = atus.stratify(
    resp, income_margin, age_margin,
    row_var=ROW_VAR, col_var=COL_VAR,
)

print(f'Building sequences (cell width = {T} minutes)...')
# modal downsampling on the intermediate integer codes (ties -> lowest code);
# no state_order needed since the states are not the final string labels
atus_seq = atus.build_sequences(diaries, T=T)
# align rows to atus_meta (weights and cluster labels are used positionally downstream)
atus_seq = atus_seq.loc[atus_meta['TUCASEID'].values]
# collapse intermediate codes -> final alphabet
atus_seq = atus_seq.map(lambda c: COLLAPSE_MAP[c])

print(f'Clustering sequences into {K} behavioural types...')
atus_metrics, cluster_results, medoid_thresholds = atus.cluster_sequences(
    atus_seq,
    weights=atus_meta[WEIGHT_COL],
    K=K,
    sequence_metric_specs=SEQUENCE_METRIC_SPECS,
    seed=SEED,
)

atus_meta['cluster_label'] = cluster_results['labels']
print('  cluster distribution:', atus_meta['cluster_label'].value_counts().sort_index().to_dict())

Loading ATUS diaries...
Stratifying respondents based on ACS targets...
auto-derived ATUS income groups: ['<35k', '35k-75k', '75k-125k', '125k+']
auto-derived ATUS age groups: ['18-24', '25-44', '45-66', '67+']
Building sequences (cell width = 30 minutes)...
Clustering sequences into 4 behavioural types...
  cluster distribution: {0: 265, 1: 325, 2: 1034, 3: 409}


# 3. Build mobility-ATUS distance matrix

Uses `SEQUENCE_METRIC_SPECS` and `MOB_SEQUENCE_COLS` from config. 

`distance_to_atus` calculates the sequence metrics based on the spec, aligns the mobility metrics to atus metrics, then calculate cosine distance matrix based on the specified subset of the metrics.


In [10]:
# GEOID must be read as a string to keep its leading zero
mob_seq = pd.read_csv(MOBILITY_SEQ_FILE, index_col=0, dtype={'GEOID': str})
mob_seq.index.name = 'user_id'   # synthetic ids from the file's row index
display(mob_seq.head())

dist_df = mobility.distance_to_atus(
    mob_seq, atus_metrics,
    sequence_metric_specs=SEQUENCE_METRIC_SPECS,
    sequence_cols=MOB_SEQUENCE_COLS,
    geoid_col='GEOID',
)
print('dist_df shape:', dist_df.shape)
display(dist_df.head())

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,GEOID
user_id,,,,,,,,,,,,,,,,,,,,,
0,Home,Home,Home,Home,Home,Home,Home,Home,Unspecified place,Work,...,Home,Home,Home,Home,Home,Home,Home,Home,Home,040030016011
1,Home,Home,Home,Home,Home,Home,Home,Home,Home,Home,...,Home,Home,Home,Home,Home,Home,Home,Home,Home,040138139002
2,Home,Home,Home,Home,Work,Work,Work,Work,Work,Work,...,Home,Home,Home,Home,Home,Home,Home,Home,Home,040131032174
3,Home,Home,Home,Home,Home,Home,POI,POI,POI,POI,...,Home,Home,Home,Home,Home,Home,Home,Home,Home,040134202161
4,Home,Home,Home,Home,Home,Home,POI,POI,POI,POI,...,Home,Home,Home,Home,Home,Home,Home,Home,Home,040131172002


aligned feature spaces: padded 0 col(s) into ATUS, 0 into mobility; active feature_subset='all'
dist_df shape: (5000, 2034)


TUCASEID,20040707040543,20040707040558,20040807040006,20040807040542,20040807040553,20040807040556,20040807040557,20040808040005,20040808040558,20040808040561,...,20191211191081,20191211191492,20191211191575,20191211191769,20191211192139,20191211192394,20191212190533,20191212191319,20191212192093,GEOID
user_id,,,,,,,,,,,,,,,,,,,,,
0,0.084274,0.243008,0.521804,0.502112,0.374391,0.356172,0.238204,0.539659,0.247963,0.385155,...,0.353286,0.588263,0.499282,0.228986,0.371027,0.272957,0.461884,0.373101,0.521804,040030016011
1,0.194804,0.102014,0.186688,0.172750,0.047304,0.006055,0.141027,0.201036,0.144622,0.042718,...,0.006094,0.600082,0.171856,0.117177,0.054463,0.105277,0.106818,0.056697,0.186688,040138139002
2,0.238728,0.330131,0.278737,0.259811,0.202224,0.217715,0.329362,0.294980,0.333682,0.235268,...,0.206491,0.697477,0.257428,0.242661,0.247818,0.164520,0.294980,0.221327,0.278737,040131032174
3,0.163875,0.002343,0.226488,0.220550,0.112710,0.095638,0.120598,0.237550,0.106178,0.167646,...,0.104415,0.482806,0.222073,0.186415,0.156689,0.201736,0.149255,0.130449,0.226488,040134202161
4,0.228844,0.083280,0.418411,0.383607,0.268716,0.290952,0.182572,0.443687,0.205246,0.272666,...,0.265379,0.566751,0.375707,0.227486,0.285451,0.362817,0.444277,0.285389,0.418411,040131172002


# 4. Cluster assignment + user filtering

Assigns each mobility user-day to one of K behavioural clusters via kNN voting with medoid-distance thresholds (users beyond the ATUS-derived 99th-percentile distance from their nearest medoid are left unassigned = -1), then drops any users whose home CBG is outside the ACS coverage.

In [11]:
assigned_labels = prep.assign_mobility_clusters(
    dist_df, atus_meta,
    cluster_label_col='cluster_label',
    k=KNN_K, threshold=KNN_THRESHOLD,
    medoid_indices=cluster_results['medoids'],
    medoid_thresholds=medoid_thresholds,
)

unit_ids, home_cbgs, assigned_labels, n_dropped = prep.filter_valid_users(
    dist_df.reset_index(),
    acs_cbg_distr,
    assigned_labels,
    geoid_col='GEOID',
    id_col='user_id',
)

print(f"Users retained: {len(home_cbgs)}")
print(f"Users dropped (missing CBG): {n_dropped}")
print("Proportion of assigned cluster labels:")
pd.Series(assigned_labels).value_counts(normalize=True).sort_index().rename("proportion")

100%|██████████| 5000/5000 [00:00<00:00, 86961.38it/s]

1367 rows have user GEOID missing from ACS GEOID (or null), e.g. ['040030016011' '040179648002' '040120201003' '040190005007'
 '040019702001']
Unique missing GEOIDs (excluding null): 686
Users retained: 3633
Users dropped (missing CBG): 1367
Proportion of assigned cluster labels:


-1    0.011561
 0    0.135150
 1    0.161574
 2    0.504266
 3    0.187448
Name: proportion, dtype: float64

# 5. Run calibration

This step preps the ACS marginal targets and the ATUS conditional cluster distribution, then instantiates a `Calibrator` object. 

Run `create_weights` in the chosen mode which returns a `CalibrationResult` with methods to get weights and metadata in various formats.


In [12]:
acs_targets = prep.prep_acs_targets(
    income_margin, 
    age_margin,
    row_var=ROW_VAR, 
    col_var=COL_VAR,
)

atus_target = prep.prep_atus_target(
    atus_meta,
    row_var=ROW_VAR, 
    col_var=COL_VAR,
    cluster_label_col='cluster_label',
    weight_col=WEIGHT_COL,
    num_row_cats=len(acs_targets['row_cats']),
    num_col_cats=len(acs_targets['col_cats']),
    num_clusters=K,
)

calibrator = Calibrator(
    unit_ids=unit_ids,
    home_cbgs=home_cbgs,
    assigned_cluster_labels=assigned_labels,
    acs_cbg_probs_df=acs_cbg_distr,
    acs_row_var_name=ROW_VAR,
    acs_col_var_name=COL_VAR,
    acs_row_cats=acs_targets['row_cats'],
    acs_col_cats=acs_targets['col_cats'],
    acs_row_margin=acs_targets['row_margin'],
    acs_col_margin=acs_targets['col_margin'],
    target_pop_tot=acs_targets['target_pop_tot'],
    atus_target_table=atus_target,
    num_replicates=NUM_REPLICATES,
    seed=SEED,
)

results = calibrator.create_weights(mode=CALIBRATION_MODE)

Weights created, returning CalibrationResult object.
Use all_final_weights() to obtain final weights.
Use to_df() to obtain replicate-specific stage1 and stage2 weights with metadata.
Use to_long_df() to obtain all replicate weights with metadata.



# 6. Inspect + save

A few quick sanity checks, plus two commented save options: a full pickle (everything) or a lighter long-format parquet (weights + metadata only).


In [13]:
print('main final weights sum:', int(results.all_final_weights()['weight_final_0'].sum()))
print('target population total:', results.target_pop_tot)
results.all_final_weights()

main final weights sum: 3708271
target population total: 3708148


,unit_id,weight_final_0,weight_final_1,weight_final_2,weight_final_3,weight_final_4,weight_final_5,weight_final_6,weight_final_7,weight_final_8,...,weight_final_41,weight_final_42,weight_final_43,weight_final_44,weight_final_45,weight_final_46,weight_final_47,weight_final_48,weight_final_49,weight_final_50
0,1,608,765,771,848,573,948,592,708,969,...,867,761,841,865,862,940,821,842,921,1131
1,2,1792,1680,1523,1617,1992,1584,2340,1347,1847,...,1946,1340,2158,1995,1579,1692,1522,728,1539,2192
2,3,1198,832,1148,1246,1001,725,592,808,640,...,851,1483,841,840,948,868,844,698,1177,810
3,4,1404,1926,1562,1578,951,494,1282,2115,1057,...,981,1202,641,1165,1221,569,1417,906,510,1556
4,6,1053,1432,878,1103,951,1231,853,1357,1085,...,909,1861,1289,1508,1220,1550,619,2258,510,826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3628,4992,528,1655,1449,1805,462,497,281,632,523,...,849,702,1664,492,1726,917,505,598,637,1943
3629,4994,974,1400,556,511,1788,497,1879,491,2057,...,889,530,502,763,1726,506,684,792,472,513
3630,4995,1024,892,849,848,870,958,749,841,969,...,816,894,896,756,758,831,804,1056,922,856
3631,4996,1094,890,1029,1211,1001,1139,921,1079,957,...,976,957,1169,1274,973,1227,1148,1198,1018,1009


In [14]:
print('first 5 rows of the main-replicate df:')
results.to_df(replicate_id=0).head()

first 5 rows of the main-replicate df:


,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,1,1138,608,2,0,8,demographic_behavioral
1,2,998,1792,1,1,5,demographic_behavioral
2,3,976,1198,3,3,15,demographic_behavioral
3,4,1055,1404,1,0,4,demographic_behavioral
4,6,1009,1053,1,2,6,demographic_behavioral


In [15]:
print('first 5 rows of the all-replicate long df:')
results.to_long_df().head()

first 5 rows of the all-replicate long df:


,replicate_id,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,0,1,1138,608,2,0,8,demographic_behavioral
1,0,2,998,1792,1,1,5,demographic_behavioral
2,0,3,976,1198,3,3,15,demographic_behavioral
3,0,4,1055,1404,1,0,4,demographic_behavioral
4,0,6,1009,1053,1,2,6,demographic_behavioral


In [ ]:
# Full pickle (all weights + full metadata as CalibrationResult object):
# results.save(OUTPUT_DIR / f'calibration_{CBSA_CODE}.pkl')

# Long-format parquet (weights from all replicates + metadata as long df):
# results.to_long_df().to_parquet(OUTPUT_DIR / f'weights_{CBSA_CODE}.parquet')